# v9c CrossJEPA — Method 2 (modality → modality)

**Total compute**: ~30–40 hours on A100 across two phases:
  - Phase A: pretrain 4 per-modality I-JEPA teachers (~6–8h each, sequential)
  - Phase B: train the Method 2 cross-modality JEPA (~10–12h)

**Inputs you need uploaded to Drive once:**
1. `MyDrive/colab_bundle.zip` — source code (already there from Method 1)
2. `MyDrive/crossjepa_data.zip` — 3D NIfTI bundle (already there from Method 1)
3. `MyDrive/dataset_v9c_modality_teachers.zip` — NEW: per-modality healthy 2D slices extracted from BraTS-2021 by `scripts/build_brats_healthy_slices.py` (~2 GB)

If you don't have (3), run `scripts/build_brats_healthy_slices.py` locally first (needs the BraTS NIfTI volumes; produces 4 subdirs of healthy 256x256 PNGs).

Compute requirements: Colab Pro+ A100 (80 GB ideally, 40 GB OK).

Pipeline:
1. Mount Drive + verify GPU
2. Install deps (`nibabel`, `smp`, `timm`)
3. Unzip source + teacher slices
4. Phase A: pretrain 4 per-modality I-JEPA teachers (loop)
5. Unzip BraTS NIfTI for Phase B
6. Phase B: train Method 2 CrossJEPA model

## 1. Mount Drive + verify GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi | head -10

## 2. Install dependencies

In [ ]:
%pip install -q nibabel==5.* huggingface_hub>=0.20 segmentation-models-pytorch>=0.3.3 timm>=0.9.16

## 3. Unzip source + teacher-slice bundles

In [ ]:
import os, sys
BUNDLE = '/content/drive/MyDrive/colab_bundle.zip'
TEACHER_DATA = '/content/drive/MyDrive/dataset_v9c_modality_teachers.zip'
DEST = '/content/neurolens'
TDEST = '/content/teacher_data'
!rm -rf {DEST} {TDEST}
!mkdir -p {DEST} {TDEST}
!unzip -q -o {BUNDLE} -d {DEST}
!unzip -q -o {TEACHER_DATA} -d {TDEST}
sys.path.insert(0, DEST); os.chdir(DEST)
import subprocess
for m in ('T1', 'T1c', 'T2', 'FLAIR'):
    n = subprocess.run(['bash', '-c', f'find {TDEST}/{m} -name "*.png" | wc -l'], capture_output=True, text=True).stdout.strip()
    print(f'  teacher {m}: {n} healthy 2D slices')

## 4. Phase A — pretrain 4 per-modality I-JEPA teachers

Each modality gets its own ~6–8h pretraining job. We loop them sequentially in a single notebook session. Checkpoints save to Drive every epoch so a disconnect doesn't lose progress.

Order: T1 → T1c → T2 → FLAIR. If your session times out before all 4 finish, re-run this cell — each trainer has `--resume auto` and skips modalities whose `last.pt` is already saved.

In [ ]:
import os, subprocess
for m in ('T1', 'T1c', 'T2', 'FLAIR'):
    out_dir = f'/content/drive/MyDrive/v9c_modality_teachers/{m}'
    final = f'{out_dir}/last.pt'
    if os.path.exists(final):
        # crude completion check: if 50 epochs done in training.log, skip
        log = f'{out_dir}/training.log'
        if os.path.exists(log) and '[done]' in open(log).read():
            print(f'[skip] {m}: already trained (last.pt + [done] in log)')
            continue
    print(f'\n=== Training teacher: {m} ===')
    !python src/train_v9b_stage1_jepa.py --data_dir {os.path.join('/content/teacher_data', m)} --output_dir {out_dir} --epochs 50 --batch_size 16 --num_workers 4 --amp --resume auto --checkpoint_every_steps 200

## 5. Unzip BraTS NIfTI for Phase B

Phase B needs the full 3D BraTS volumes (Mod2ModDataset streams 4-modality volumes per patient). This is the same `crossjepa_data.zip` used by Method 1.

In [ ]:
DATA_BUNDLE = '/content/drive/MyDrive/crossjepa_data.zip'
DATA_DEST = '/content/data'
!rm -rf {DATA_DEST}
!mkdir -p {DATA_DEST}
!unzip -q -o {DATA_BUNDLE} -d {DATA_DEST}
import subprocess
n_b = subprocess.run(['bash', '-c', f'ls {DATA_DEST}/brats | wc -l'], capture_output=True, text=True).stdout.strip()
print(f'  unzipped {n_b} BraTS patient directories')

## 6. Phase B — train Method 2 CrossJEPA

Per-modality teachers feed into the Mod2ModModel. Phase B asks the encoder to predict any held-out modality's teacher embedding given a subset of the other modalities.

In [ ]:
!python src/train_v9c_method2_mod2mod.py --brats_root /content/data/brats --teachers_dir /content/drive/MyDrive/v9c_modality_teachers --output_dir /content/drive/MyDrive/v9c_crossjepa_method2 --image_size 256 --batch_size 8 --epochs 30 --lr 2e-4 --num_workers 2 --amp --resume auto --checkpoint_every_steps 200

## Total wall-clock estimate

| Phase | Time on A100 |
|---|---|
| 4 teachers (sequential) | ~24–32 h |
| Method 2 (30 epochs, BraTS, batch=8) | ~10–15 h |
| **Total** | **~35–47 h** |

Run across multiple sessions. Resume is automatic for every step.